# TP Seance 3 - Transfer Learning & Fine-Tuning
## ResNet-50 sur Oxford Flowers 102

**Module :** Deep Learning Avance - Golden Collar
**GPU recommande :** Google Colab (Runtime -> T4 GPU)

| Partie | Theme |
|--------|-------|
| 1 | Chargement & exploration du dataset |
| 2 | Feature Extraction (backbone gele) |
| 3 | Fine-Tuning (degel partiel, LR faible) |
| 4 | Augmentation avancee (Mixup) |
| 5 | Interpretabilite (Grad-CAM) |
| 6 | (Bonus) VGG-16 vs ResNet-50 |

> **Fil conducteur :** trois briques reutilisables - `build_transfer_model(backbone=...)`,
> `train(...)`, `plot_history(...)`. Chaque modele a son **propre nom de variable**
> (pas d'ecrasement), et les comparaisons ne changent **qu'un facteur a la fois**.

## Installation

In [ ]:
!pip install -q tensorflow tensorflow-datasets matplotlib numpy scikit-learn

In [ ]:
import os, datetime
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50, VGG16
from tensorflow.keras.applications.resnet50 import preprocess_input  # mode 'caffe' (idem VGG16)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, TensorBoard

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow :", tf.__version__)
print("GPU :", tf.config.list_physical_devices('GPU') or "aucun (CPU)")

LOG_DIR = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(LOG_DIR, exist_ok=True)

---
## Partie 1 - Chargement & exploration

**Oxford Flowers 102** : 102 classes, ~8000 images. Petit et riche -> ideal pour le Transfer Learning.

In [ ]:
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_CLASSES  = 102
EPOCHS_FE    = 15      # feature extraction
EPOCHS_FT    = 10      # fine-tuning

In [ ]:
(ds_train_raw, ds_val_raw, ds_test_raw), ds_info = tfds.load(
    'oxford_flowers102', split=['train', 'validation', 'test'],
    as_supervised=True, with_info=True)

print("Train      :", ds_info.splits['train'].num_examples)
print("Validation :", ds_info.splits['validation'].num_examples)
print("Test       :", ds_info.splits['test'].num_examples)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def augment(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.2)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + 20, IMG_SIZE + 20)
    image = tf.image.random_crop(image, [IMG_SIZE, IMG_SIZE, 3])
    return image, label

def resize_only(image, label):
    return tf.image.resize(image, [IMG_SIZE, IMG_SIZE]), label

def preprocess(image, label):
    # preprocess_input attend des valeurs 0-255 (pas de /255) ; il soustrait la moyenne ImageNet
    return preprocess_input(tf.cast(image, tf.float32)), label

def make_ds(ds_raw, training=False):
    ds = ds_raw.map(augment if training else resize_only, num_parallel_calls=AUTOTUNE)
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

ds_train = make_ds(ds_train_raw, training=True)
ds_val   = make_ds(ds_val_raw)
ds_test  = make_ds(ds_test_raw)

def denormalize(img):
    """Inverse de preprocess_input (mode caffe) pour la visualisation."""
    mean = np.array([103.939, 116.779, 123.68])      # BGR
    img = img + mean
    img = img[..., ::-1]                              # BGR -> RGB
    return np.clip(img / 255.0, 0, 1)

print("Pipelines prets.")

In [ ]:
# Apercu des images brutes
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for ax, (image, label) in zip(axes.flat, ds_train_raw.take(18)):
    ax.imshow(image.numpy()); ax.set_title(f"classe {label.numpy()}", fontsize=8); ax.axis('off')
plt.suptitle('Oxford Flowers 102', fontweight='bold'); plt.tight_layout(); plt.show()

### Questions - Partie 1
- **Q1.1** Pourquoi normaliser avec les stats **ImageNet** et non celles du dataset ?
- **Q1.2** ~80 images/classe : pourquoi le Transfer Learning est-il pertinent ici ?
- **Q1.3** Quelles augmentations seraient **inappropriees** pour des fleurs ?

*Vos reponses :* ...

---
## Les briques reutilisables

In [ ]:
def build_transfer_model(backbone='resnet50', dropout=0.5):
    """Backbone pre-entraine gele + tete GAP -> BN -> Dropout -> softmax."""
    Base = ResNet50 if backbone == 'resnet50' else VGG16
    base = Base(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(base.input, out, name=f'{backbone}_transfer'), base


def train(model, name, lr, epochs, ds_tr, ds_va, mixup=False):
    """Compile + callbacks + fit. mixup=True -> labels soft (categorical)."""
    loss = 'categorical_crossentropy' if mixup else 'sparse_categorical_crossentropy'
    acc  = 'categorical_accuracy'     if mixup else 'accuracy'
    model.compile(optimizer=Adam(lr), loss=loss, metrics=[acc])
    cbs = [EarlyStopping(monitor=f'val_{acc}', patience=5, restore_best_weights=True, verbose=1),
           ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0),
           TensorBoard(log_dir=os.path.join(LOG_DIR, name), histogram_freq=0)]
    return model.fit(ds_tr, validation_data=ds_va, epochs=epochs, callbacks=cbs, verbose=1)


def acc_key(history):
    return next(k for k in ('accuracy', 'categorical_accuracy') if k in history.history)


def plot_history(history, title=""):
    k = acc_key(history)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(history.history['loss'], label='train'); a1.plot(history.history['val_loss'], '--', label='val')
    a1.set_title('Loss'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
    a2.plot(history.history[k], label='train'); a2.plot(history.history[f'val_{k}'], '--', label='val')
    a2.set_title('Accuracy'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
    if title: fig.suptitle(title, fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()
    print(f"Meilleure val accuracy : {max(history.history[f'val_{k}']):.4f}")

---
## Partie 2 - Feature Extraction (backbone gele)

Backbone ResNet-50 **completement gele** ; on entraine seulement la tete. LR standard `1e-3`.

In [ ]:
model_fe, base = build_transfer_model('resnet50', dropout=0.5)
trainable = sum(tf.keras.backend.count_params(w) for w in model_fe.trainable_weights)
total = model_fe.count_params()
print(f"Total params      : {total:,}")
print(f"Entrainables      : {trainable:,} ({100*trainable/total:.1f}%)")
print(f"Geles (backbone)  : {total-trainable:,} ({100*(total-trainable)/total:.1f}%)")

In [ ]:
history_fe = train(model_fe, 'feature_extraction', lr=1e-3, epochs=EPOCHS_FE,
                   ds_tr=ds_train, ds_va=ds_val)
plot_history(history_fe, "Feature Extraction - ResNet-50")
results_fe = model_fe.evaluate(ds_test, verbose=0)
print(f"Test accuracy (FE) : {results_fe[1]*100:.2f}%")

### Questions - Partie 2
- **Q2.1** Pourquoi un LR `1e-3` est-il acceptable ici mais pas en fine-tuning ?
- **Q2.2** Avantages du **GAP** vs les 3 couches FC de VGG ?
- **Q2.3** Ratio params geles/actifs : impact sur la vitesse vs un entrainement *from scratch* ?

*Vos reponses :* ...

---
## Partie 3 - Fine-Tuning (degel partiel, LR faible)

On **continue** l'entrainement du modele de la Partie 2 (meme objet `model_fe`), en degelant les
couches profondes avec un LR tres petit (`1e-5`). Les **BatchNorm restent gelees** (stats ImageNet).

> *Pour aller plus loin :* LR differentiels (LR plus faible pour les couches basses) - cf. slides.

In [ ]:
base.trainable = True
FREEZE_UNTIL = 143      # couches < 143 gelees ; >= 143 entrainables

for i, layer in enumerate(base.layers):
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False              # garder les statistiques pre-entrainees
    else:
        layer.trainable = i >= FREEZE_UNTIL

trainable = sum(tf.keras.backend.count_params(w) for w in model_fe.trainable_weights)
total = model_fe.count_params()
print(f"Apres degel (>= {FREEZE_UNTIL}) : entrainables {trainable:,} ({100*trainable/total:.1f}%)")

In [ ]:
history_ft = train(model_fe, 'fine_tuning', lr=1e-5, epochs=EPOCHS_FT,
                   ds_tr=ds_train, ds_va=ds_val)
model_ft = model_fe                          # nom explicite : modele fine-tune (meilleur modele)
plot_history(history_ft, "Fine-Tuning - ResNet-50")
results_ft = model_ft.evaluate(ds_test, verbose=0)
print(f"Test accuracy (FT) : {results_ft[1]*100:.2f}%")
print(f"Gain FT vs FE      : {(results_ft[1]-results_fe[1])*100:+.2f} points")

### Questions - Partie 3
- **Q3.1** Qu'est-ce que le *catastrophic forgetting* et comment l'eviter ?
- **Q3.2** Pourquoi geler les BatchNorm pendant le fine-tuning (surtout petit batch) ?
- **Q3.3** *(exp)* Changez `FREEZE_UNTIL` (100, 50) : impact sur performance et temps ?

*Vos reponses :* ...

---
## Partie 4 - Augmentation avancee : Mixup

**Mixup** (Zhang et al., 2018) : interpolation lineaire de deux images **et** de leurs labels.
On garde **la meme tete** qu'en Partie 2 pour isoler l'effet de Mixup.

In [ ]:
def mixup_batch(images, labels, alpha=0.3, num_classes=NUM_CLASSES):
    batch = tf.shape(images)[0]
    g1 = tf.random.gamma([1], alpha); g2 = tf.random.gamma([1], alpha)
    lam = g1 / (g1 + g2); lam = tf.maximum(lam[0], 1.0 - lam[0])
    idx = tf.random.shuffle(tf.range(batch))
    y  = tf.one_hot(tf.cast(labels, tf.int32), num_classes)
    y2 = tf.gather(y, idx)
    mixed_x = lam * images + (1 - lam) * tf.gather(images, idx)
    mixed_y = lam * y + (1 - lam) * y2
    return mixed_x, mixed_y

# Visualisation
batch_x, batch_y = next(iter(ds_train))
mx, my = mixup_batch(batch_x, batch_y, alpha=0.4)
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for i in range(4):
    axes[0, i].imshow(denormalize(batch_x[i].numpy())); axes[0, i].set_title(f"classe {batch_y[i].numpy()}", fontsize=8); axes[0, i].axis('off')
    top = tf.argsort(my[i], direction='DESCENDING')[:2].numpy()
    p   = tf.sort(my[i], direction='DESCENDING')[:2].numpy()
    axes[1, i].imshow(denormalize(mx[i].numpy()))
    axes[1, i].set_title(f"C{top[0]}({p[0]:.2f}) + C{top[1]}({p[1]:.2f})", fontsize=8); axes[1, i].axis('off')
plt.suptitle('Mixup : original (haut) vs mixe (bas)', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
def one_hot(image, label):
    return image, tf.one_hot(tf.cast(label, tf.int32), NUM_CLASSES)

# Train avec Mixup applique au niveau du batch ; val/test en one-hot
ds_train_mix = (ds_train_raw.map(augment, num_parallel_calls=AUTOTUNE)
                .map(preprocess, num_parallel_calls=AUTOTUNE).shuffle(1024).batch(BATCH_SIZE)
                .map(lambda x, y: mixup_batch(x, y, 0.3), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))
ds_val_oh  = ds_val.map(one_hot,  num_parallel_calls=AUTOTUNE)
ds_test_oh = ds_test.map(one_hot, num_parallel_calls=AUTOTUNE)

model_mix, _ = build_transfer_model('resnet50', dropout=0.5)   # meme tete que Partie 2
history_mix = train(model_mix, 'mixup', lr=1e-3, epochs=EPOCHS_FE,
                    ds_tr=ds_train_mix, ds_va=ds_val_oh, mixup=True)
plot_history(history_mix, "Feature Extraction + Mixup")
results_mix = model_mix.evaluate(ds_test_oh, verbose=0)
print(f"Test accuracy (Mixup FE) : {results_mix[1]*100:.2f}%")
print(f"vs FE classique          : {(results_mix[1]-results_fe[1])*100:+.2f} points")

### Questions - Partie 4
- **Q4.1** Pourquoi `categorical_crossentropy` (et non `sparse_...`) avec Mixup ?
- **Q4.2** Effet de `alpha` quand alpha -> 0 ? quand alpha = 1 ?
- **Q4.3** *(bonus)* Implementez **CutMix** : differences attendues vs Mixup ?

*Vos reponses :* ...

---
## Partie 5 - Interpretabilite : Grad-CAM

On visualise les zones d'attention du **meilleur modele** (le modele fine-tune `model_ft`).

In [ ]:
def gradcam(model, image, layer_name='conv5_block3_out', class_idx=None):
    grad_model = Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(image, training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        score = preds[:, class_idx]
    grads = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))           # importance par filtre
    heatmap = tf.nn.relu(tf.squeeze(conv_out[0] @ weights[..., None]))
    heatmap = heatmap.numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap, class_idx, float(preds[0][class_idx])

def overlay(image_rgb, heatmap, alpha=0.45):
    hm = tf.image.resize(heatmap[..., None], [IMG_SIZE, IMG_SIZE]).numpy().squeeze()
    colored = plt.get_cmap('jet')(hm)[:, :, :3]
    return np.clip(alpha * colored + (1 - alpha) * image_rgb, 0, 1)

print("Grad-CAM pret.")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
images, labels = next(iter(ds_test))
for ax, i in zip(axes.flat, range(9)):
    hm, pred, prob = gradcam(model_ft, images[i:i+1])
    ax.imshow(overlay(denormalize(images[i].numpy()), hm))
    tag = "correct" if pred == labels[i].numpy() else "erreur"
    ax.set_title(f"pred C{pred} ({prob*100:.0f}%) {tag} | vrai C{labels[i].numpy()}", fontsize=8)
    ax.axis('off')
plt.suptitle('Grad-CAM - zones d attention', fontweight='bold'); plt.tight_layout(); plt.show()

### Questions - Partie 5
- **Q5.1** Sur un cas d'erreur, la heatmap aide-t-elle a comprendre ?
- **Q5.2** Que verrait-on avec une couche plus precoce (`conv3_block4_out`) ? Essayez.
- **Q5.3** Comment Grad-CAM aide-t-il a detecter un biais de dataset ?

*Vos reponses :* ...

---
## Partie 6 (Bonus) - VGG-16 vs ResNet-50

Comparaison **equitable** : meme strategie (Feature Extraction), meme nombre d'epochs.

In [ ]:
model_vgg, _ = build_transfer_model('vgg16', dropout=0.5)
history_vgg = train(model_vgg, 'vgg16', lr=1e-3, epochs=EPOCHS_FE, ds_tr=ds_train, ds_va=ds_val)
results_vgg = model_vgg.evaluate(ds_test, verbose=0)

print(f"\n{'Modele (Feature Extraction)':<28}{'params':>14}{'test acc':>10}")
print('-' * 52)
print(f"{'VGG-16':<28}{model_vgg.count_params():>14,}{results_vgg[1]*100:>9.2f}%")
print(f"{'ResNet-50':<28}{model_fe.count_params():>14,}{results_fe[1]*100:>9.2f}%")

---
## Synthese

In [ ]:
def best_val(h): return max(h.history[f'val_{acc_key(h)}'])

rows = [("ResNet-50 Feature Extraction", best_val(history_fe),  results_fe[1]),
        ("ResNet-50 Fine-Tuning",        best_val(history_ft),  results_ft[1]),
        ("ResNet-50 Mixup FE",           best_val(history_mix), results_mix[1]),
        ("VGG-16 Feature Extraction",    best_val(history_vgg), results_vgg[1])]

print(f"{'Experience':<32}{'val acc':>10}{'test acc':>10}")
print('-' * 52)
for name, v, t in rows:
    print(f"{name:<32}{v*100:>9.2f}%{t*100:>9.2f}%")
print(f"\nGain Feature Extraction -> Fine-Tuning : {(results_ft[1]-results_fe[1])*100:+.2f} points")

### A retenir
1. Toujours normaliser avec `preprocess_input` du backbone (stats ImageNet).
2. Deux etapes : Feature Extraction (LR grand, backbone gele) puis Fine-Tuning (LR tres petit).
3. Geler les BatchNorm pendant le fine-tuning.
4. Mixup ameliore la generalisation avec peu de donnees.
5. Grad-CAM valide que le modele "regarde" la fleur, pas le fond.
6. ResNet-50 : moins de parametres que VGG-16, meilleures performances.

### Pour aller plus loin
- EfficientNetB3 / ResNet50V2 ; LR differentiels ; CutMix ; Vision Transformer (cf. Seance 5).

### Visualiser avec TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/